# **Week 2 Deliverables**


### **1.   Task 1: MetaData and Image Audit**
        * Devyani Lenin - MetaData Audit
        * Check the HAM10000 metadata CSV for missing values, duplicate image IDs.
        * Document how problems are handled without editing raw data.
        * Output: Metadata audit code and a summary of findings.


        * Rishita Singh - Image Audit
        * Make sure images from both image folders can be located and opened.
        * Identify any images referenced in metadata that are missing or unreadable.
        * Document how problems are handled without editing raw data.
        * Output: Image audit code and a summary of image problems.


### **2.   Task 2 - Split Generation and Verification**
        * Parin Vakati - Split Generation
        * After the Modeling & Experiments group agrees on the split  proportions and manifest format, write the code that generates train, validation, and test manifests using a fixed random seed.
        * Make sure images with the same lesion_id stay together.
        * Output: Split-generation Python code.

        * Yaden Gant — Split Verification
        * Verify that no lesion_id appears in multiple splits, report class counts for each split, and check that running the code again with the same seed produces identical results.
        * Output: Verification code and evidence that the splits work correctly.


### **3.   Member 5 - Final Documentation**
      * Coordinate with Modeling & Experiments and Explainability & App.  
      * Help make sure the notebooks and PR descriptions clearly identify everyone's contributions.
      * Document the agreed seven-class mapping, split format, dependencies, data paths, commands, findings, and remaining blockers.
      * Output: Documentation under docs/week2/ and coordination notes.

  



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **1. Code that loads the CSV file**


1. Link to data: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/DBW86T

2. I have shared a folder in Google drive with the required file path for everyone. If you run the cell below, it will ask you to 'Connect to Google Drive'. You can go ahead and connect,grant all permissions,  and you should be able to load the dataset.

3. Here is the link to the Google Drive for anyone in the project outside of our Group: https://drive.google.com/drive/folders/1gA-YqreIhWDNeQhykp4QWuSLDQbXYWuc?usp=sharing

4. **IF YOU GET AN ERROR WHEN LOADING FILE** ->  If it does not work, go to your Google drive and right click on the folder. You should see an 'Organize' option there, from which you can click on 'Add shortcut' and select the location as 'My Drive'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import os


file_path = (
    "/content/drive/MyDrive/"
    "PixelPathologyFall2026/data/raw/"
    "HAM10000/HAM10000_metadata.csv"
)


df = pd.read_csv(file_path)

print(f"Records: {len(df)}")
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Records: 10015
     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp   
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear   

        dataset  
0  vidir_modern  
1  vidir_modern  
2  vidir_modern  
3  vidir_modern  
4  vidir_modern  


## **Task 1 - MetaData Audit**

* Only the age column is missing vales. Since none of the lesion_id, image_id, and dx columns are missing values, we have decided to not create a new dataset where those 57 records are removed. Instead, we will address them as missing, and remove them later if necessary.

In [ ]:
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

df.info()
df.head()

Total rows: 10015
Total columns: 8
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10015 entries, 0 to 10014
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   lesion_id     10015 non-null  object 
 1   image_id      10015 non-null  object 
 2   dx            10015 non-null  object 
 3   dx_type       10015 non-null  object 
 4   age           9958 non-null   float64
 5   sex           10015 non-null  object 
 6   localization  10015 non-null  object 
 7   dataset       10015 non-null  object 
dtypes: float64(1), object(7)
memory usage: 626.1+ KB


,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,vidir_modern
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,vidir_modern
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,vidir_modern
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,vidir_modern
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,vidir_modern


In [ ]:
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)


print("\nTotal missing values:", missing_values.sum())

Missing values per column:
lesion_id        0
image_id         0
dx               0
dx_type          0
age             57
sex              0
localization     0
dataset          0
dtype: int64

Total missing values: 57


In [ ]:
duplicate_count = df["image_id"].duplicated().sum()

print("Number of duplicate image IDs:", duplicate_count)


duplicate_rows = df[
    df["image_id"].duplicated(keep=False)
]

display(duplicate_rows)

important_columns = ["image_id", "lesion_id", "dx"]

missing_important = df[
    df[important_columns].isnull().any(axis=1)
]

print("Rows missing important information:", len(missing_important))

display(missing_important)

Number of duplicate image IDs: 0


,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset


Rows missing important information: 0


,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset


In [ ]:
audit_summary = {
    "Total metadata rows": len(df),
    "Missing values (total)": int(df.isnull().sum().sum()),
    "Rows with missing values": int(df.isnull().any(axis=1).sum()),
    "Duplicate image IDs": int(df["image_id"].duplicated().sum()),
    "Rows missing important fields": len(missing_important)
}

print("HAM10000 Metadata Audit Summary")
print("-" * 35)

for check, result in audit_summary.items():
    print(f"{check}: {result}")

HAM10000 Metadata Audit Summary
-----------------------------------
Total metadata rows: 10015
Missing values (total): 57
Rows with missing values: 57
Duplicate image IDs: 0
Rows missing important fields: 0


In [ ]:
# counting "unknown" values in each column
unknown_counts = df.apply(
    lambda col: col.astype("string").str.strip().str.lower().eq("unknown").sum()
)

print("Unknown values per column:")
print(unknown_counts[unknown_counts > 0])

Unknown values per column:
sex              57
localization    234
dtype: int64


In [ ]:
required_columns = ["image_id", "lesion_id", "dx"]

for col in required_columns:
    unknown_count = (
        df[col].astype("string").str.strip().str.lower() == "unknown"
    ).sum()

    print(f"{col}: {unknown_count} unknown values")

image_id: 0 unknown values
lesion_id: 0 unknown values
dx: 0 unknown values


## **Task 1 - Image Audit**

In [ ]:
from pathlib import Path
from PIL import Image #PIL verifies image integrity

def perform_image_audit(metadata_csv_path, folder_part1, folder_part2, output_csv=None):
    """
    Audits images referenced in HAM10000 metadata across two image folders.
    Checks for file existence, extension, and corrupt/unreadable files.

    """
    # 1. Load metadata
    df = pd.read_csv(metadata_csv_path)
    print(f"Loaded metadata with {len(df)} total image records.")

    # Convert paths to Path objects
    p1 = Path(folder_part1)
    p2 = Path(folder_part2)

    # 2. Build quick lookup maps for files present on disk
    # HAM10000 images usually have .jpg extension
    disk_files = {}
    for folder in [p1, p2]:
        if folder.exists():
            for img_file in folder.glob("*.jpg"):
                # Map image_id (stem without extension) to full path
                disk_files[img_file.stem] = img_file
        else:
            print(f"Warning: Folder '{folder}' does not exist.")

    print(f"Found {len(disk_files)} total .jpg files across both folders.")

    # 3. Perform audit for each metadata row
    audit_results = []

    for idx, row in df.iterrows():
        img_id = row['image_id']

        # Check if file exists on disk
        if img_id in disk_files:
            file_path = disk_files[img_id]

            # Check image readability and integrity
            try:
                with Image.open(file_path) as img:
                    img.verify()  # Verifies file header and integrity without loading full image into RAM

                status = "VALID"
                err_msg = ""
            except Exception as e:
                status = "CORRUPT"
                err_msg = str(e)
        else:
            file_path = None
            status = "MISSING"
            err_msg = "File not found in provided image folders"

        audit_results.append({
            'image_id': img_id,
            'lesion_id': row.get('lesion_id', None),
            'dx': row.get('dx', None),
            'found_path': str(file_path) if file_path else None,
            'audit_status': status,
            'error_details': err_msg
        })

    audit_df = pd.DataFrame(audit_results)

    # 4. Generate Summary Statistics
    summary = audit_df['audit_status'].value_counts()
    print("\n" + "="*40)
    print("           IMAGE AUDIT SUMMARY           ")
    print("="*40)
    for status_type, count in summary.items():
        pct = (count / len(df)) * 100
        print(f" - {status_type}: {count} ({pct:.2f}%)")
    print("="*40)

    # Filter out problematic rows
    if not audit_df[audit_df['audit_status'] != "VALID"].empty:
        print(f"\nIdentified {len(audit_df[audit_df['audit_status'] != "VALID"])} problematic entries.")
        print(audit_df[audit_df['audit_status'] != "VALID"][['image_id', 'audit_status', 'error_details']].head(10))
    else:
        print("\nAll images referenced in the metadata are valid and readable!")

    # 5. Export audited version without modifying raw source data
    if output_csv:
        valid_df = audit_df[audit_df['audit_status'] == "VALID"]
        valid_df.to_csv(output_csv, index=False)
        print(f"\nCleaned metadata saved to: {output_csv}")

    return audit_df, audit_df[audit_df['audit_status'] != "VALID"]

#Example execution path setup:
# audit_df, issues = perform_image_audit(
#     metadata_csv_path="HAM10000_metadata.csv",
#     folder_part1="HAM10000_images_part_1",
#     folder_part2="HAM10000_images_part_2",
#     output_csv="cleaned_HAM10000_metadata.csv"
# )

## **Task 2 - Split Generation**

In [ ]:
def make_splits(meta, seed=42):
  x = meta['image_id']
  y = meta['dx']

  split1 = GroupShuffleSplit(test_size=0.3, random_state=seed)

  train_idx, temp_idx = next(split1.split(x, y, groups=meta['lesion_id']))

  x_train, x_temp = x.iloc[train_idx], x.iloc[temp_idx]
  y_train, y_temp = y.iloc[train_idx], y.iloc[temp_idx]

  split2 = GroupShuffleSplit(test_size=0.5, random_state=seed)

  val_idx, test_idx = next(split2.split(x_temp, y_temp, groups=meta['lesion_id'].iloc[temp_idx]))

  x_val, x_test = x_temp.iloc[val_idx], x_temp.iloc[test_idx]
  y_val, y_test = y_temp.iloc[val_idx], y_temp.iloc[test_idx]

  return x_train, x_val, x_test, y_train, y_val, y_test

x_train, x_val, x_test, y_train, y_val, y_test = make_splits(df, seed=42)

## **Task 2 - Split Verification**

In [ ]:
import pandas as pd

# Make sure to always load CSV file and split generation file before running this

splits = {"train": x_train, "val": x_val, "test": x_test}
SEED = 42

# One table: image_id, split, lesion_id, dx (joined by image_id, not position)
long = pd.concat(
    [pd.DataFrame({"image_id": ids.values, "split": name}) for name, ids in splits.items()],
    ignore_index=True,
).merge(df[["image_id", "lesion_id", "dx"]], on="image_id", how="left", validate="m:1")

# Coverage: every image exactly once, no missing lesion_ids
assert long["image_id"].is_unique, "an image appears in more than one split"
assert set(long["image_id"]) == set(df["image_id"]), "images missing or extra"
assert long["lesion_id"].notna().all(), "null lesion_id"

# lesion_id check
n_splits = long.groupby("lesion_id")["split"].nunique()
assert (n_splits == 1).all(), f"{(n_splits > 1).sum()} lesion_ids appear in multiple splits"
print("PASS: no lesion_id appears in more than one split")

# class count check
counts = pd.crosstab(long["split"], long["dx"]).reindex(["train", "val", "test"])
print(counts)
print(counts.div(counts.sum(axis=1), axis=0).mul(100).round(1))

# same seed check
def assignment(seed):
    return [sorted(s) for s in make_splits(df, seed)[:3]]   # train, val, test

assert assignment(SEED) == assignment(SEED), "same seed gave different splits"
assert assignment(SEED) == [sorted(s) for s in splits.values()], "notebook splits != make_splits(seed)"
print("PASS: same seed reproduces identical splits")

PASS: no lesion_id appears in more than one split
dx     akiec  bcc  bkl  df  mel    nv  vasc
split                                      
train    207  381  741  89  770  4718    96
val       57   65  206  19  156   991    25
test      63   68  152   7  187   996    21
dx     akiec  bcc   bkl   df   mel    nv  vasc
split                                         
train    3.0  5.4  10.6  1.3  11.0  67.4   1.4
val      3.8  4.3  13.6  1.3  10.3  65.2   1.6
test     4.2  4.6  10.2  0.5  12.5  66.7   1.4
PASS: same seed reproduces identical splits
